# Classificacao de Imagens com CNN (Rede Neural Convolucional)

## Visao Geral

Este notebook demonstra como criar, treinar e fazer deploy de um modelo **CNN (Convolutional Neural Network)** para classificacao de imagens do dataset Fashion-MNIST.

### Por que usar CNN para imagens?

As CNNs sao especialmente eficazes para processamento de imagens porque:
- **Preservam relacoes espaciais** entre pixels
- **Detectam padroes locais** (bordas, texturas, formas)
- **Sao invariantes a translacao** (reconhecem objetos em qualquer posicao)
- **Requerem menos parametros** que redes totalmente conectadas

### Arquitetura da CNN

```
Input (28x28x1) 
    -> Conv2D (32 filtros) -> ReLU -> MaxPool
    -> Conv2D (64 filtros) -> ReLU -> MaxPool
    -> Flatten
    -> Dense (128) -> ReLU -> Dropout
    -> Dense (10) -> Softmax
```

In [ ]:
class FashionCNN(nn.Module):
    """
    Rede Neural Convolucional para classificacao de imagens Fashion-MNIST.
    
    Arquitetura:
    - 2 blocos convolucionais (Conv -> BatchNorm -> ReLU -> MaxPool)
    - 2 camadas fully connected com Dropout
    """
    
    def __init__(self, num_classes=10):
        super(FashionCNN, self).__init__()
        
        # Bloco Convolucional 1
        # Input: (1, 28, 28) -> Output: (32, 14, 14)
        self.conv1 = nn.Conv2d(
            in_channels=1,      # Imagem grayscale
            out_channels=32,    # 32 filtros
            kernel_size=3,      # Filtro 3x3
            padding=1           # Manter dimensao
        )
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14
        
        # Bloco Convolucional 2
        # Input: (32, 14, 14) -> Output: (64, 7, 7)
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        
        # Bloco Convolucional 3 (opcional, para mais profundidade)
        # Input: (64, 7, 7) -> Output: (128, 3, 3)
        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 7x7 -> 3x3
        
        # Camadas Fully Connected
        # Flatten: 128 * 3 * 3 = 1152
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        # Bloco 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Bloco 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # Bloco 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully Connected
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        
        return x

# Criar modelo
model = FashionCNN(num_classes=10).to(device)

# Mostrar arquitetura
print("Arquitetura da CNN:")
print("=" * 60)
print(model)
print("=" * 60)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parametros: {total_params:,}")
print(f"Parametros treinaveis: {trainable_params:,}")

## 3. Treinamento do Modelo

Vamos usar:
- **Otimizador**: Adam com learning rate 0.001
- **Loss**: CrossEntropyLoss (para classificacao multiclasse)
- **Scheduler**: ReduceLROnPlateau (reduz LR quando loss estagna)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Treina uma epoca"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Avalia o modelo"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), correct / total


def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    """Loop completo de treinamento"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, verbose=True
    )
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_acc = 0.0
    
    print("Iniciando treinamento...")
    print("=" * 70)
    
    for epoch in range(epochs):
        # Treinar
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Avaliar
        val_loss, val_acc = evaluate(model, test_loader, criterion, device)
        
        # Scheduler
        scheduler.step(val_loss)
        
        # Salvar historico
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Salvar melhor modelo
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_cnn_model.pth')
        
        # Log
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.6f}")
    
    print("=" * 70)
    print(f"Melhor acuracia de validacao: {best_acc:.4f}")
    
    return history

In [ ]:
# Treinar o modelo
EPOCHS = 10
history = train_model(model, train_loader, test_loader, epochs=EPOCHS, lr=0.001)

In [ ]:
# Visualizar historico de treinamento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss durante Treinamento')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Acc', marker='o')
ax2.plot(history['val_acc'], label='Val Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Acuracia durante Treinamento')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAcuracia final de treino: {history['train_acc'][-1]:.4f}")
print(f"Acuracia final de validacao: {history['val_acc'][-1]:.4f}")

## 4. Avaliacao Detalhada

Vamos analisar a performance do modelo por classe usando matriz de confusao e metricas.